<a href="https://colab.research.google.com/github/xyt556/I-GUIDE-GeoAI-Education/blob/main/notebooks/08-image-translation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 图像翻译

## 介绍

之前的教程侧重于从图像中提取标签或掩模。图像翻译采用不同的方法：它不生成标签，而是通过学习从一个图像域到另一个图像域的映射，同时保留空间结构来生成新图像。

图像到图像翻译涵盖了许多遥感任务，包括传感器翻译（例如，SAR 到光学）、时间间隙填充、着色和合成数据生成。所有这些都具有共同的结构：模型将输入图像 $x$ 从域 A 映射到域 B 中的输出图像 $y$，其中这些域在分辨率、光谱内容、传感器类型或视觉风格上有所不同。

其中，超分辨率是其中最具实际重要性的一项。Sentinel-2 等任务每五天提供 10 米的免费全球覆盖，而商业卫星以更高的成本实现亚米级分辨率。超分辨率使用深度学习来提高免费可用图像的空间分辨率，部分弥补了这一差距。

本教程重点介绍使用 LDSR-S2 潜在扩散模型将 Sentinel-2 图像从 10 米分辨率增强到 2.5 米分辨率。您将下载 Sentinel-2 场景，对单个补丁和更大区域运行超分辨率，可视化增强输出，并计算不确定性图。

## 学习目标

完成本教程后，您将能够：

- 解释什么是图像到图像翻译以及它在遥感中的主要应用
- 描述超分辨率问题以及它对卫星图像分析的重要性
- 解释潜在扩散模型如何通过编码、去噪和解码执行超分辨率
- 下载并检查 Sentinel-2 多光谱图像以进行超分辨率处理
- 运行单补丁超分辨率以将 128x128 像素区域在每个空间维度上增强四倍
- 比较低分辨率输入和超分辨率输出以评估增强质量
- 使用随机正向传播计算和解释每像素不确定性图
- 使用重叠混合的平铺推理处理大于单个补丁的区域

## 图像翻译基础

### 什么是图像到图像翻译？

图像到图像翻译将图像从一个域转换为另一个域，生成新图像而不是分类标签。输入和输出共享空间结构，但在分辨率、光谱内容或传感器模态等某些属性上有所不同。

像 Pix2Pix 这样的配对翻译框架使用条件 GANs 来学习对齐图像对之间的映射，而 CycleGAN 将此扩展到非配对设置。在遥感中，这些架构已被用于 SAR 到光学翻译、跨传感器协调和地图生成。

对于地理空间应用，最重要的图像翻译类别包括：

1. **超分辨率**：增强空间分辨率，例如将 10 米的 Sentinel-2 像素转换为 2.5 米像素
2. **传感器翻译**：在成像模态之间转换，例如从 SAR 数据生成光学图像
3. **时间合成**：为没有直接观测的日期生成图像，填补云或重访时间表造成的空白
4. **光谱增强**：向缺少光谱带的图像添加光谱带，例如从 RGB 输入预测多光谱内容
5. **合成数据生成**：为缺少标记数据的区域创建逼真的训练图像

### 遥感超分辨率

超分辨率 (SR) 从一个或多个低分辨率输入重建高分辨率图像，产生比传感器原生提供的更精细的空间细节。如果模型能够可靠地将免费可用的 10 米图像增强到接近 2.5 米的质量，它将在全球范围内以零额外数据成本实现详细的空间分析。

单图像超分辨率 (SISR) 是最具挑战性的变体，因为模型必须推断未直接观察到的精细细节。这个问题是不适定的：许多高分辨率图像在下采样时可能会产生相同的低分辨率观测结果。早期的基于 CNN 的方法最小化了像素级误差，但产生了过于平滑的输出。生成模型（GANs 和最近的扩散模型）通过学习生成清晰、逼真的输出解决了这个问题。

### 用于超分辨率的潜在扩散模型

扩散模型通过逐步向训练图像添加噪声（前向阶段）然后学习逐步逆转这种损坏（反向阶段）来生成图像。在低分辨率输入上调节反向过程，使模型能够生成一致的高分辨率输出和可信的精细细节。

潜在扩散模型 (LDM) 通过在压缩表示空间而不是直接在全分辨率像素上工作来降低计算成本。编码器将图像映射到低维潜在空间，扩散在此处操作，解码器将结果映射回像素空间。

LDSR-S2 模型将潜在扩散应用于 Sentinel-2 超分辨率。它在四个光谱带（红色、绿色、蓝色和近红外）上操作，并执行 4 倍空间上采样，将 10 米像素转换为 2.5 米像素。每个 128x128 补丁的工作流程是：

1. **编码**：低分辨率补丁（128x128，4 个波段）被压缩成潜在表示
2. **去噪**：扩散过程在潜在空间中运行指定数量的采样步骤，以低分辨率输入为条件
3. **解码**：去噪后的潜在表示被解码回 4 倍分辨率的像素空间（512x512，4 个波段）

扩散模型的一个关键优势是不确定性量化。使用不同的随机种子多次运行模型会产生略微不同的输出，这些变化的标准差提供了每像素不确定性估计。

## 安装

取消注释以下行以安装所需的包。

In [ ]:
# %pip install -U "geoai-py[extra]"

## 导入库

In [ ]:
import geoai
import numpy as np
import rasterio as rio
from matplotlib import pyplot as plt

## 下载样本数据

我们使用诺克斯维尔、田纳西州上空的 Sentinel-2 Level-2A 子集。图像包含四个 10 米波段（红、绿、蓝和近红外），存储为 `uint16` 大气层底部 (BOA) 反射率值，范围为 0 到 10,000。

In [ ]:
url = "https://data.source.coop/opengeos/geoai/S2C-MSIL2A-20250920T162001-Knoxville.tif"
s2_path = geoai.download_file(url)

### 检查输入数据

处理前检查图像尺寸、坐标参考系和像素分辨率。

In [ ]:
with rio.open(s2_path) as src:
    print(f"Bands: {src.count}")
    print(f"Size: {src.width} x {src.height}")
    print(f"CRS: {src.crs}")
    print(f"Resolution: {src.res[0]:.2f} m")
    print(f"Dtype: {src.dtypes[0]}")

```text
波段: 4
尺寸: 2874 x 1623
CRS: EPSG:3857
分辨率: 10.00 m
数据类型: uint16
```

图像有四个波段，分辨率为 10 米。`uint16` 数据类型和 0 到 10,000 的值范围是 Sentinel-2 L2A 产品的标准，其中 1,000 对应于 0.1 (10%) 的表面反射率。

## 可视化输入 RGB 合成图

使用红、绿、蓝波段显示真彩色合成图，并进行百分位对比度拉伸，将第 2 和第 98 百分位值映射到 0 和 1。

In [ ]:
with rio.open(s2_path) as src:
    rgb = src.read([1, 2, 3]).astype(np.float32)

for i in range(3):
    band = rgb[i]
    p2, p98 = np.percentile(band, (2, 98))
    rgb[i] = (band - p2) / (p98 - p2)
rgb = np.clip(rgb, 0, 1)

fig, ax = plt.subplots(figsize=(12, 7))
ax.imshow(rgb.transpose(1, 2, 0))
ax.set_title("Sentinel-2 RGB Composite (10 m)")
ax.set_axis_off()
plt.tight_layout()
plt.show()

## 单补丁超分辨率

LDSR-S2 模型处理 128x128 像素的补丁。`window` 参数以像素坐标指定 `(row_offset, col_offset, height, width)`。模型对补丁进行编码，运行去噪扩散，并以 4 倍分辨率解码结果，生成 512x512 像素的 2.5 米输出。

In [ ]:
sr_output = "sr_output.tif"
sr_image, _ = geoai.super_resolution(
    input_lr_path=s2_path,
    output_sr_path=sr_output,
    rgb_nir_bands=[1, 2, 3, 4],
    window=(700, 1300, 128, 128),
    sampling_steps=100,
)

In [ ]:
print(f"Input shape:  (4, 128, 128) at 10 m")
print(f"Output shape: {sr_image.shape} at 2.5 m")

```text
输入形状:  (4, 128, 128) 在 10 米
输出形状: (4, 512, 512) 在 2.5 米
```

### 比较低分辨率和超分辨率

使用 `plot_sr_comparison` 并排显示低分辨率输入和超分辨率输出的 RGB 比较。

In [ ]:
geoai.plot_sr_comparison(s2_path, sr_output, bands=[1, 2, 3])
plt.show()

验证输出 GeoTIFF 具有正确的空间参考和 2.5 米的像素大小。

In [ ]:
with rio.open(sr_output) as src:
    print(f"SR Bands: {src.count}")
    print(f"SR Size: {src.width} x {src.height}")
    print(f"SR CRS: {src.crs}")
    print(f"SR Resolution: {src.res[0]:.2f} m")

```text
SR 波段: 4
SR 大小: 512 x 512
SR CRS: EPSG:3857
SR 分辨率: 2.50 m
```

## 不确定性估计

基于扩散的超分辨率可以量化每像素不确定性。使用不同的随机种子多次运行模型会产生略微不同的输出，这些变化的标准差产生不确定性图。

设置 `compute_uncertainty=True` 并使用 `n_variations` 控制要运行多少个随机正向传播。更多的变化会产生更平滑的估计，但会增加计算时间。

In [ ]:
sr_unc_output = "sr_with_uncertainty.tif"
unc_output = "uncertainty.tif"
sr_image2, uncertainty = geoai.super_resolution(
    input_lr_path=s2_path,
    output_sr_path=sr_unc_output,
    output_uncertainty_path=unc_output,
    rgb_nir_bands=[1, 2, 3, 4],
    window=(700, 1300, 128, 128),
    compute_uncertainty=True,
    n_variations=5,
    sampling_steps=100,
)

### 可视化不确定性图

使用伪彩色可视化显示不确定性图。红色/黄色区域表示更高的不确定性（不同种子间的输出变化更大），而绿色区域表示更高的置信度。

高不确定性通常出现在尖锐边界和具有复杂精细纹理的区域。开放水域或裸土等均质区域通常显示低不确定性。

In [ ]:
geoai.plot_sr_uncertainty(unc_output)
plt.show()

## 大型区域的平铺推理

对于大于 128x128 像素的区域，函数会自动将输入平铺成重叠的补丁，独立处理每个补丁，并使用线性混合拼接结果以防止出现可见的接缝。

`patch_size` 参数设置图块大小，`overlap` 控制相邻补丁共享的像素数量。更大的重叠会产生更平滑的过渡，但会增加计算时间。

In [ ]:
sr_large = "sr_large.tif"
sr_large_img, _ = geoai.super_resolution(
    input_lr_path=s2_path,
    output_sr_path=sr_large,
    rgb_nir_bands=[1, 2, 3, 4],
    window=(700, 1300, 256, 256),
    patch_size=128,
    overlap=16,
    sampling_steps=100,
)

In [ ]:
print(f"Input shape:  (4, 256, 256) at 10 m")
print(f"Output shape: {sr_large_img.shape} at 2.5 m")

```text
输入形状:  (4, 256, 256) 在 10 米
输出形状: (4, 1024, 1024) 在 2.5 米
```

### 比较较大区域的结果

比较 256x256 区域的低分辨率输入和拼接后的超分辨率输出。

In [ ]:
geoai.plot_sr_comparison(s2_path, sr_large, bands=[1, 2, 3])
plt.show()

### 交互式分屏地图比较

使用分屏地图交互式地比较超分辨率输出和高分辨率底图图像。请注意，底图可能来自不同的采集日期，因此不匹配可能反映时间差异以及模型错误。

In [ ]:
geoai.create_split_map(
    left_layer=sr_large, right_layer="Esri.WorldImagery", left_args={"vmax": 0.3}
)

## 局限性与注意事项

超分辨率模型增强了视觉细节，但它们不能恢复真实的地面信息。精细特征是统计预测，而不是直接观测。建筑物轮廓可能看起来更清晰，但与真实轮廓不匹配，道路边缘可能在不存在的地方被幻觉。这对于定量分析来说最为重要。从超分辨率输出测量建筑物面积或划分田地边界可能会引入系统误差。不确定性图有助于标记低置信度区域，但低不确定性并不能保证正确性。

超分辨率模型还带有其训练数据的偏差，并且可能不会保留植被指数计算或变化检测所需的辐射关系。超分辨率最好被视为一种视觉增强工具，而不是真正高分辨率图像的替代品。

## 关键要点

1. **图像翻译生成新图像而非标签**，学习图像域之间的映射，同时保留空间结构。

2. **超分辨率通过从较粗糙的输入推断可信的精细结构，增强了超出传感器原生能力的细节。**

3. **潜在扩散模型通过编码到潜在空间、在低分辨率输入条件下进行去噪，以及解码回像素空间来执行超分辨率。**

4. **在潜在空间中工作降低了计算成本**，同时在标准硬件上保持了高质量结果。

5. **LDSR-S2 模型对 Sentinel-2 图像执行 4 倍超分辨率**，处理 128x128 补丁中的四个波段以生成 512x512 像素的 2.5 米输出。

6. **不确定性图通过计算多次随机前向传播的标准差来量化每像素模型置信度。**

7. **带重叠混合的平铺推理处理任意大小的场景**，通过处理重叠补丁并无缝混合它们。

8. **输出 GeoTIFF 保留完整的地理参考**，地理变换自动调整以适应更精细的像素大小。